# Train and Test: Scoring a Model Honestly

Last time we trained a linear regression and scored it with `model.score(X, y)`. That
grades the model on the **same rows it learned from**, which flatters it.

This notebook shows why that matters, and what to do instead: **hold some rows back**.

> **Note:** `nyc_rent.csv` is **invented (synthetic) data** for teaching. The patterns
> are realistic, but the rows are not real listings.

## The data

Same 300 NYC apartments as before. We predict `rent`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

apts = pd.read_csv("nyc_rent.csv")
apts.head()

,size_sqft,bedrooms,subway_min,dist_manhattan_mi,building_age,days_listed,rent
0,349,0,19,7.5,16,16,1000
1,426,0,13,10.3,25,41,1325
2,675,1,16,3.2,42,30,2950
3,880,2,15,5.3,107,38,3300
4,645,1,12,11.7,62,41,1550


## The tempting mistake

Fit on every row, then score on every row. The model has already seen every answer.

In [2]:
X = apts[["size_sqft"]]
y = apts["rent"]

model = LinearRegression().fit(X, y)
print("R-squared on the training data:", round(model.score(X, y), 3))

R-squared on the training data: 0.757


That looks respectable. But it is an **open book exam**: we asked the model about
rows it had already studied.

## Hold some rows back

`train_test_split` shuffles the rows and sets a portion aside. We **fit** on the
training rows and **score** on the test rows, which the model never saw.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)      # hold back 20% of the rows

print("training rows:", len(X_train), " test rows:", len(X_test))

checked = LinearRegression().fit(X_train, y_train)   # learn here
print("train R-squared", round(checked.score(X_train, y_train), 3))
print("test  R-squared", round(checked.score(X_test,  y_test),  3))   # grade there

training rows: 240  test rows: 60
train R-squared 0.769
test  R-squared 0.703


The test score is **lower**. That gap is the part of the training score that was
flattery rather than skill.

Two arguments worth knowing:

- `test_size=0.2` holds back 20 percent of the rows. 0.2 to 0.3 is typical.
- `random_state=0` fixes the shuffle, so you get the same split every run. Without it
  your scores wobble a little each time you re-run the cell.

**Aside:** you will also see a third set, **validation**, used to compare models and
tune settings, so the test set stays sealed until the very end.

## More features, still honest

Add the useful columns. Fit on train, report test.

In [4]:
good = ["size_sqft", "bedrooms", "subway_min", "dist_manhattan_mi", "building_age"]

X_train, X_test, y_train, y_test = train_test_split(
    apts[good], apts["rent"], test_size=0.2, random_state=0)

big = LinearRegression().fit(X_train, y_train)
print("train R-squared", round(big.score(X_train, y_train), 3))
print("test  R-squared", round(big.score(X_test,  y_test),  3))

train R-squared 0.952
test  R-squared 0.944


Both scores rose and they sit close together: more useful features, a better fit,
and no sign of memorizing.

## Now the experiment: add features that mean nothing

`junk` columns are pure random noise, unrelated to rent by construction. Watch what
each score does as we add more of them.

In [5]:
def train_and_score(n_junk, seed=0, test_size=0.3):
    "Fit on train with n_junk random columns added; return (train R2, test R2)."
    rng = np.random.default_rng(seed)
    junk = pd.DataFrame({f"junk{i}": rng.normal(size=len(apts)) for i in range(n_junk)})
    X = pd.concat([apts[good], junk], axis=1)

    Xtr, Xte, ytr, yte = train_test_split(
        X, apts["rent"], test_size=test_size, random_state=1)
    m = LinearRegression().fit(Xtr, ytr)
    return m.score(Xtr, ytr), m.score(Xte, yte)


print(f"{'junk':>6} {'TRAIN R2':>10} {'TEST R2':>10}")
for k in [0, 1, 5, 20, 50]:
    tr, te = train_and_score(k)
    print(f"{k:>6} {tr:>10.4f} {te:>10.4f}")

  junk   TRAIN R2    TEST R2
     0     0.9526     0.9382
     1     0.9533     0.9380
     5     0.9542     0.9356
    20     0.9570     0.9349
    50     0.9659     0.9079


Read the **direction** of each column, not just the size.

- **Train goes up.** Adding columns can only help the fit on rows the model studied.
  Given random noise, it will find a faint fake pattern and use it.
- **Test goes down.** Those patterns were never real, so they do not survive contact
  with new data.

The two scores pulling apart is **overfitting**. If you only ever looked at the training
score, useless features would look harmless or even helpful.

## How far can this go?

The damage depends on the **ratio** of features to training rows. We have 300 rows, so
a 30 percent test split leaves **210 training rows**. Watch what happens as the number
of features climbs toward that number.

In [6]:
print(f"{'junk':>6} {'features':>9} {'train rows':>11} {'TRAIN R2':>10} {'TEST R2':>12}")
for k in [50, 100, 150, 190, 205]:
    tr, te = train_and_score(k)
    print(f"{k:>6} {k+len(good):>9} {210:>11} {tr:>10.4f} {te:>12.4f}")

  junk  features  train rows   TRAIN R2      TEST R2
    50        55         210     0.9659       0.9079
   100       105         210     0.9760       0.8663
   150       155         210     0.9868       0.7484
   190       195         210     0.9970       0.0599
   205       210         210     1.0000     -18.3871


That last row is the cliff. With **210 features and 210 training rows**, the model
has exactly enough knobs to pass through every training point: train R-squared is
**1.0000**, a flawless score, and the test score is **catastrophically negative**.

**What does a negative R-squared mean?** R-squared compares your model against the
simplest possible baseline: always guessing the average rent. A score of 0 means you
matched that baseline; below 0 means you did **worse than guessing the average every
time**. The model did not learn a weak pattern here. It learned nothing transferable at
all, and confidently reported nonsense.

This is the same lesson as before, turned up loud: **a perfect training score is a
warning sign, not an achievement.**

## The rule of thumb

Think in **rows per training feature**:

| rows per feature | what happens |
| --- | --- |
| about 1 | the model memorizes; predictions collapse |
| 10 to 20 | generally safe |
| 50 or more | junk features are nearly free |

With plenty of rows, a few useless columns cost almost nothing. With few rows, they are
dangerous. Most real datasets you pull from NYC Open Data are closer to the small end
than you would like.

## The takeaway

- **Split your data.** Fit on train, report the **test** score.
- **Watch both numbers.** A rising training score with a falling test score is
  overfitting, and it is invisible if you only look at one.
- **A perfect training score is a red flag.**
- **Ridge** and **Lasso** are far more resilient to junk features than plain
  `LinearRegression`, but they are a safety net, not a substitute for thinking about
  which features belong in your model.